# Visualizations and simple models
Two datasets: housing prices for exploratory visuals and a quick linear regression on height/weight.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
sns.set(style='whitegrid', context='notebook')


In [ ]:
housing = pd.read_csv('Housing.csv')
print(f'shape: {housing.shape}')
print(housing.dtypes)
housing.head()


In [ ]:
num_cols = ['price', 'lotsize', 'bedrooms', 'bathrms', 'stories', 'garagepl']
housing[num_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(housing['price'], kde=True, ax=axes[0], color='teal', bins=35)
axes[0].set_title('Price distribution')
sns.boxplot(x=housing['airco'], y=housing['price'], ax=axes[1], palette='husl')
axes[1].set_title('Price vs airco flag')
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.regplot(x='lotsize', y='price', data=housing, ax=axes[0], scatter_kws={'s':16, 'alpha':0.45}, line_kws={'color': 'darkorange'})
axes[0].set_title('Lot size vs price')
sns.regplot(x='bedrooms', y='price', data=housing, ax=axes[1], x_jitter=0.1, scatter_kws={'color': '#7B68EE'})
axes[1].set_title('Bedrooms vs price')
plt.tight_layout()


In [ ]:
corr = housing[num_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu', square=True)
plt.title('Correlation matrix for numeric features')
plt.tight_layout()


In [ ]:
binary_cols = ['driveway', 'recroom', 'fullbase', 'gashw', 'airco', 'prefarea']
encoded_binary = housing[binary_cols].apply(lambda s: s.str.lower().map({'yes': 1, 'no': 0}))
feature_block = pd.concat([housing[num_cols], encoded_binary], axis=1)
scaled = pd.DataFrame(StandardScaler().fit_transform(feature_block), columns=feature_block.columns)
scaled.head()


## Linear regression for weight vs height


In [ ]:
growth = pd.read_csv('rost_ves.csv').rename(columns={'Рост (см)': 'height_cm', 'Вес (кг)': 'weight_kg'})
growth.head()


In [ ]:
X = growth[['height_cm']]
y = growth['weight_kg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)
linreg = LinearRegression()
linreg.fit(X_train, y_train)
pred = linreg.predict(X_test)
print({'MAE': mean_absolute_error(y_test, pred), 'R2': r2_score(y_test, pred)})


In [ ]:
height_grid = np.linspace(growth['height_cm'].min(), growth['height_cm'].max(), 120).reshape(-1, 1)
trend = linreg.predict(height_grid)
plt.figure(figsize=(6, 4))
plt.scatter(growth['height_cm'], growth['weight_kg'], s=30, alpha=0.7, label='observations')
plt.plot(height_grid, trend, color='crimson', label='linear fit')
plt.xlabel('Height (cm)')
plt.ylabel('Weight (kg)')
plt.title('Linear regression on growth data')
plt.legend()
plt.tight_layout()


In [ ]:
sample_heights = np.array([[165], [172], [185]])
pd.DataFrame({
    'height_cm': sample_heights.flatten(),
    'predicted_weight': linreg.predict(sample_heights).round(2)
})
